# Ablation Study: Candidate Pool Size (num_beams / num_candidates)

Addresses Examiner 2's comment: *"Mengapa hanya 4 kandidat. Jelaskan kalau 3 dan 5 kandidat. Mengapa itu tidak dilakukan."*

Runs generation + NLI reranking with `num_candidates` (= `num_beams`) in {2, 3, 5, 6}, at the fixed, already-confirmed α = 0.7, on the **validation split** (methodologically consistent with how α itself was selected — see `03_ablation_study_alpha.ipynb`'s fix note — rather than on the test split).

The k=4 configuration does **not** need to be re-run: it is already available from `results/alpha_selection_validation_results.json`'s α=0.7 row (that sweep was run with `num_candidates=4` fixed), so this notebook only needs k ∈ {2, 3, 5, 6}.

**Cost note:** cost scales roughly linearly with k for both beam-search decoding and the k NLI forward passes per document. Running all four k values on the full 10,964-document validation split is possible but expensive; `SAMPLE_SIZE` below defaults to 2000 documents (fixed random seed, so it's the same subsample every run) — raise it toward the full validation set if your GPU quota allows.

## 1. Check GPU & Install

In [ ]:
!pip install -q transformers rouge-score

import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

## 2. Paths and Sample Selection

In [ ]:
import json
import math
import random
import time
from pathlib import Path

MODEL_PATH = "/kaggle/input/datasets/madedwikibudilaksana/output-notebook/results/thesis_pipeline/outputs/bart-baseline"
VALID_FILE = "/kaggle/input/datasets/madedwikibudilaksana/output-notebook/results/thesis_pipeline/data/processed/valid.jsonl"
# Update these two paths if your Kaggle input mount differs (see 03_ablation_study_alpha.ipynb).

TEXT_COLUMN = "article"
SUMMARY_COLUMN = "summary"

SAMPLE_SIZE = 2000  # raise toward len(valid_rows) if GPU quota allows; keep fixed SEED for reproducibility
SEED = 42
ALPHA = 0.7  # already confirmed via validation-split sweep, see results/alpha_selection_validation_results.json
K_VALUES = [2, 3, 5, 6]  # k=4 is already available, see the note above — no need to rerun it here

print("Model path:", MODEL_PATH)
print("Validation file:", VALID_FILE)
print("config.json exists:", Path(MODEL_PATH, "config.json").exists())
print("valid.jsonl exists:", Path(VALID_FILE).exists())

with open(VALID_FILE, "r", encoding="utf-8") as f:
    valid_rows = [json.loads(l) for l in f if l.strip()]

random.seed(SEED)
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(valid_rows):
    valid_rows = random.sample(valid_rows, SAMPLE_SIZE)

print(f"Using {len(valid_rows)} validation documents (SAMPLE_SIZE={SAMPLE_SIZE}, seed={SEED})")

## 3. Load Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification

NLI_MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
device = "cuda" if torch.cuda.is_available() else "cpu"

bart_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(device).eval()
print("BART loaded")

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME).to(device).eval()
id2label = {int(k): v.lower() for k, v in nli_model.config.id2label.items()}
ent_idx = next(i for i, l in id2label.items() if "entail" in l)
con_idx = next(i for i, l in id2label.items() if "contrad" in l)
print("NLI loaded")

## 4. Run Generation + Reranking for Each k

In [ ]:
import statistics
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

def run_for_k(k, rows):
    r1, r2, rl, ents, cons = [], [], [], [], []
    t0 = time.time()
    for i, row in enumerate(rows):
        document = row[TEXT_COLUMN]
        with torch.inference_mode():
            inputs = bart_tokenizer(document, truncation=True, max_length=256, return_tensors="pt")
            inputs = {kk: v.to(device) for kk, v in inputs.items()}
            generated = bart_model.generate(
                **inputs,
                max_length=128, min_length=32,
                num_beams=k, num_return_sequences=k,
                length_penalty=1.0, early_stopping=True,
                output_scores=True, return_dict_in_generate=True,
            )
        texts = bart_tokenizer.batch_decode(generated.sequences, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        gen_scores = generated.sequences_scores.tolist()

        best = None
        for text, gen_score in zip(texts, gen_scores):
            summary = text.strip()
            with torch.inference_mode():
                enc = nli_tokenizer(document, summary, truncation=True, max_length=512, return_tensors="pt")
                enc = {kk: v.to(device) for kk, v in enc.items()}
                probs = torch.softmax(nli_model(**enc).logits[0], dim=-1)
            ent = float(probs[ent_idx])
            con = float(probs[con_idx])
            norm_gen = math.tanh(gen_score / 10.0)
            combined = ALPHA * ent + (1 - ALPHA) * norm_gen
            if best is None or combined > best["combined"]:
                best = {"summary": summary, "entailment": ent, "contradiction": con, "combined": combined}

        s = scorer.score(row[SUMMARY_COLUMN], best["summary"])
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rl.append(s["rougeL"].fmeasure)
        ents.append(best["entailment"])
        cons.append(best["contradiction"])

        if (i + 1) % 200 == 0:
            print(f"  k={k}: {i+1}/{len(rows)}", flush=True)

    elapsed = time.time() - t0
    n = len(rows)
    return {
        "k": k,
        "n_samples": n,
        "rouge1": sum(r1) / n, "rouge2": sum(r2) / n, "rougeL": sum(rl) / n,
        "entailment": statistics.mean(ents), "contradiction": statistics.mean(cons),
        "avg_seconds_per_sample": elapsed / n,
    }

pool_results = []
for k in K_VALUES:
    print(f"\n=== Running k={k} on {len(valid_rows)} documents ===")
    result = run_for_k(k, valid_rows)
    pool_results.append(result)
    print(f"k={k} | R1={result['rouge1']:.4f} | R2={result['rouge2']:.4f} | RL={result['rougeL']:.4f} | "
          f"Ent={result['entailment']:.4f} | Con={result['contradiction']:.4f} | "
          f"{result['avg_seconds_per_sample']:.2f}s/sample")

print("\nAblation over candidate pool size complete.")

## 5. Save Results (merged with the existing k=4 reference row)

In [ ]:
# Splice in the already-available k=4 row from the validation-split alpha sweep
# (results/alpha_selection_validation_results.json, alpha=0.7 row) so the final table has all 5 k values
# without needing to regenerate k=4. Update this dict if that file's alpha=0.7 numbers change.
k4_reference_row = {
    "k": 4,
    "n_samples": None,  # was run on the full 10,964-doc validation split, not the SAMPLE_SIZE used here
    "rouge1": 0.3516, "rouge2": 0.1778, "rougeL": 0.2893,
    "entailment": 0.4545, "contradiction": 0.0102,
    "avg_seconds_per_sample": None,
    "source": "results/alpha_selection_validation_results.json (alpha=0.7 row, full validation split, not resampled)",
}

all_results = sorted(pool_results + [k4_reference_row], key=lambda r: r["k"])

output_path = Path("./results/candidate_pool_size_ablation_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump({
        "alpha": ALPHA, "sample_size": SAMPLE_SIZE, "seed": SEED,
        "note": "k=4 row is from the full validation split (different n than k=2/3/5/6, which used SAMPLE_SIZE); treat as a reference point, not a strictly like-for-like comparison unless SAMPLE_SIZE == full validation set size.",
        "results": all_results,
    }, f, indent=2)
print(f"Saved to {output_path}")

print("\n=== Copy into thesis as a new table (Tabel baru, Ablasi Ukuran Pool Kandidat) ===")
print(f"{'k':>3} | {'ROUGE-1':>8} | {'ROUGE-2':>8} | {'ROUGE-L':>8} | {'Entailment':>10} | {'Contradiction':>13} | {'s/sample':>9}")
print("-" * 75)
for r in all_results:
    spp = f"{r['avg_seconds_per_sample']:.2f}" if r["avg_seconds_per_sample"] is not None else "n/a"
    print(f"{r['k']:>3} | {r['rouge1']:>8.4f} | {r['rouge2']:>8.4f} | {r['rougeL']:>8.4f} | "
          f"{r['entailment']:>10.4f} | {r['contradiction']:>13.4f} | {spp:>9}")

## Narasi untuk disalin ke tesis (draf, sesuaikan dengan angka aktual hasil run)

Contoh kalimat untuk subbab 3.4 / 4.4.7, mengisi Examiner 2 #3:

> Untuk menguji kepekaan hasil terhadap ukuran pool kandidat, dilakukan studi ablasi tambahan dengan memvariasikan jumlah kandidat (`num_beams = num_return_sequences`) pada nilai k = 2, 3, 4, 5, dan 6, dengan bobot α = 0,7 tetap, pada [**n dokumen**, ISI] dari validation split. Hasilnya menunjukkan **[ISI: pola diminishing returns / tidak ada pola jelas / dst., sesuaikan dengan hasil]**, dengan k = 4 [**tetap kompetitif / lebih rendah dari k=X / dst., ISI**] pada skor *entailment* sekaligus mempertahankan biaya komputasi yang lebih rendah dibandingkan k = 5 atau 6 (waktu inferensi meningkat kira-kira linear terhadap k, lihat subbab 4.4.7). Berdasarkan hasil ini, k = 4 [**tetap dipertahankan sebagai / direvisi menjadi k=X sebagai**, ISI SESUAI HASIL] konfigurasi akhir penelitian ini.

**Penting:** isi bagian `[ISI]` di atas SESUAI ARAH HASIL AKTUAL setelah notebook dijalankan — jangan menyalin klaim "k=4 tetap optimal" bila hasil sebenarnya menunjukkan k lain lebih unggul. Kejujuran di sini penting, mengikuti pola yang sudah dipakai konsisten pada revisi α (subbab 4.4.3).